#Introduction
# Supplier Risk Scoring System using Machine Learning

This project develops a machine learning-based supplier evaluation system to predict delivery delay risk and rank suppliers using operational and behavioral metrics.

##Import Librarie

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import MinMaxScaler

##SECTION 3 — Load Dataset

In [2]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{\r\n  "username": "nikola02727",\r\n  "key": "KGAT_369a2a4e6f52df075390d1cdd6d63608"\r\n}'}

In [ ]:
# Configure Kaggle API
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
# Install Kaggle package

!pip install kaggle

In [5]:
# Download dataset
!kaggle datasets download -d harshmadhavan/vendor-performance-analysis

Dataset URL: https://www.kaggle.com/datasets/harshmadhavan/vendor-performance-analysis
License(s): CC0-1.0
100% 484M/484M [00:06<00:00, 81.9MB/s]



In [6]:
# Extract dataset
!unzip vendor-performance-analysis.zip

Archive:  vendor-performance-analysis.zip
  inflating: begin_inventory.csv     
  inflating: end_inventory.csv       
  inflating: inventory.db            
  inflating: purchase_prices.csv     
  inflating: purchases.csv           
  inflating: sales.csv               
  inflating: vendor_invoice.csv      
  inflating: vendor_sales_summary.csv  


In [7]:
import os
os.listdir()

['.config',
 'begin_inventory.csv',
 'end_inventory.csv',
 'vendor_sales_summary.csv',
 'purchase_prices.csv',
 'purchases.csv',
 'inventory.db',
 'kaggle.json',
 'vendor-performance-analysis.zip',
 'vendor_invoice.csv',
 'sales.csv',
 'sample_data']

In [8]:
# Load purchase dataset
purchase_df=pd.read_csv('purchases.csv')
purchase_df.head()

,InventoryId,Store,Brand,Description,Size,VendorNumber,VendorName,PONumber,PODate,ReceivingDate,InvoiceDate,PayDate,PurchasePrice,Quantity,Dollars,Classification
0,69_MOUNTMEND_8412,69,8412,Tequila Ocho Plata Fresno,750mL,105,ALTAMAR BRANDS LLC,8124,2023-12-21,2024-01-02,2024-01-04,2024-02-16,35.71,6,214.26,1
1,30_CULCHETH_5255,30,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,4,37.40,1
2,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-02,2024-01-07,2024-02-21,9.41,5,47.05,1
3,1_HARDERSFIELD_5255,1,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,6,56.10,1
4,76_DONCASTER_2034,76,2034,Glendalough Double Barrel,750mL,388,ATLANTIC IMPORTING COMPANY,8169,2023-12-24,2024-01-02,2024-01-09,2024-02-16,21.32,5,106.60,1


In [9]:
purchase.size


NameError: name 'purchase' is not defined

In [ ]:
purchase_df.info()

In [ ]:
purchase_df.describe()

# Data Preprocessing and Initial Exploration

In [ ]:
# Convert date columns to datetime format

purchase_df['PODate'] = pd.to_datetime(purchase_df['PODate'])
purchase_df['ReceivingDate'] = pd.to_datetime(purchase_df['ReceivingDate'])
purchase_df['InvoiceDate'] = pd.to_datetime(purchase_df['InvoiceDate'])
purchase_df['PayDate'] = pd.to_datetime(purchase_df['PayDate'])

In [ ]:
# Check missing values

purchase_df.isnull().sum()

In [ ]:
# Drop irrelevant columns

purchase_df = purchase_df.drop(
    columns=['Description', 'PONumber', 'Classification']
).copy()

In [ ]:
# Check invalid delivery records

invalid_dates = (
    purchase_df['ReceivingDate'] < purchase_df['PODate']
).sum()

print("Invalid delivery records:", invalid_dates)

In [ ]:
# Calculate delivery time

purchase_df['delivery_time'] = (
    purchase_df['ReceivingDate'] - purchase_df['PODate']
).dt.days

In [ ]:
# Delivery time summary

purchase_df['delivery_time'].describe()

In [ ]:
# Delivery time distribution

import matplotlib.pyplot as plt

purchase_df['delivery_time'].hist(bins=50)

plt.title('Distribution of Delivery Time')
plt.xlabel('Delivery Time (Days)')
plt.ylabel('Frequency')

plt.show()

In [ ]:
# Remove old version if exists (optional but clean)
if 'late_delivery_risk' in purchase_1.columns:
    purchase_1 = purchase_1.drop(columns=['late_delivery_risk'])

# Create vendor-based delay
vendor_avg = purchase_1.groupby('VendorNumber')['delivery_time'].mean()
purchase_1['vendor_avg_time'] = purchase_1['VendorNumber'].map(vendor_avg)

purchase_1['late_delivery_risk'] = (
    purchase_1['delivery_time'] > purchase_1['vendor_avg_time']
).astype(int)


In [ ]:
purchase_1['late_delivery_risk'].value_counts()

In [ ]:
purchase_1.groupby('late_delivery_risk')[['Quantity','PurchasePrice','Dollars']].mean()

In [ ]:
purchase_1.columns

## Feature Creation

In [ ]:
# Create vendor-level average delivery time

vendor_avg = purchase_df.groupby('VendorNumber')['delivery_time'].mean()

purchase_df['vendor_avg_time'] = (
    purchase_df['VendorNumber'].map(vendor_avg)
)

In [ ]:
# Create target variable

purchase_df['late_delivery_risk'] = (
    purchase_df['delivery_time'] >
    purchase_df['vendor_avg_time']
).astype(int)

In [ ]:
# Sort data for temporal feature engineering

purchase_df = purchase_df.sort_values(
    ['VendorNumber', 'PODate']
)

In [ ]:
# Previous delivery delay

purchase_df['vendor_prev_delay'] = (
    purchase_df.groupby('VendorNumber')['delivery_time']
    .shift(1)
)

In [ ]:
# Previous delivery delay

purchase_df['vendor_prev_delay'] = (
    purchase_df.groupby('VendorNumber')['delivery_time']
    .shift(1)
)

In [ ]:
# Previous delivery delay

purchase_df['vendor_prev_delay'] = (
    purchase_df.groupby('VendorNumber')['delivery_time']
    .shift(1)
)

In [ ]:
# Rolling average delivery performance

purchase_df['vendor_recent_avg'] = (
    purchase_df.groupby('VendorNumber')['delivery_time']
    .shift(1)
    .rolling(window=5)
    .mean()
)

In [ ]:
# Recent delay frequency

purchase_df['recent_bad_rate'] = (
    purchase_df.groupby('VendorNumber')['late_delivery_risk']
    .shift(1)
    .rolling(window=5)
    .mean()
)

In [ ]:
# Payment delay

purchase_df['payment_delay'] = (
    purchase_df['PayDate'] -
    purchase_df['InvoiceDate']
).dt.days

In [ ]:
# Processing delay

purchase_df['processing_delay'] = (
    purchase_df['InvoiceDate'] -
    purchase_df['ReceivingDate']
).dt.days

In [ ]:
# Extract temporal indicators

purchase_df['month'] = purchase_df['PODate'].dt.month

purchase_df['day_of_week'] = (
    purchase_df['PODate'].dt.dayofweek
)

In [ ]:
# Check missing values in final features

purchase_df[
    [
        'vendor_prev_delay',
        'vendor_recent_avg',
        'recent_bad_rate',
        'payment_delay',
        'processing_delay',
        'month',
        'day_of_week'
    ]
].isnull().sum()

In [ ]:
features = [
    'vendor_prev_delay',
    'vendor_recent_avg',
    'recent_bad_rate',
    'payment_delay',
    'processing_delay',
    'month',
    'day_of_week'
]

# Model Training and Evaluation

In [ ]:
# Sample dataset for efficient model training

sample = purchase_df.sample(
    200000,
    random_state=42
).fillna(0)

In [ ]:
# Create feature matrix and target variable

X = sample[features]

y = sample['late_delivery_risk']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=20,
    max_depth=10,
    n_jobs=-1,
    random_state=42
)
rf_model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [ ]:
# Random Forest predictions

y_pred_rf = rf_model.predict(X_test)

In [ ]:
# Classification report

print(classification_report(y_test, y_pred_rf))

In [ ]:
# Confusion matrix

print(confusion_matrix(y_test, y_pred_rf))

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train, y_train)

In [ ]:
# Logistic Regression predictions

y_pred_lr = lr_model.predict(X_test)

In [ ]:
# Logistic Regression evaluation

print(classification_report(y_test, y_pred_lr))

# Supplier Risk Scoring System

In [ ]:
# Generate prediction probabilities

sample['risk_score'] = (
    rf_model.predict_proba(X)[:, 1]
)

In [ ]:
sample.columns

In [ ]:
# Supplier Score
supplier_score = (
    sample.groupby('VendorNumber')['risk_score']
    .mean()
    .reset_index()
)

In [ ]:
supplier_ops = (
    sample.groupby('VendorNumber')[
        ['payment_delay', 'processing_delay']
    ]
    .mean()
    .reset_index()
)

supplier_score = supplier_score.merge(
    supplier_ops,
    on='VendorNumber'
)

In [ ]:
supplier_stability = (
    sample.groupby('VendorNumber')[
        'recent_bad_rate'
    ]
    .mean()
    .reset_index()
)

supplier_score = supplier_score.merge(
    supplier_stability,
    on='VendorNumber'
)

In [ ]:
supplier_env = (
    sample.groupby('VendorNumber')[
        'vendor_prev_delay'
    ]
    .mean()
    .reset_index()
)

supplier_score = supplier_score.merge(
    supplier_env,
    on='VendorNumber'
)

In [ ]:
price_std = (
    purchase_df.groupby('VendorNumber')[
        'PurchasePrice'
    ]
    .std()
    .reset_index()
)

price_std.columns = [
    'VendorNumber',
    'price_variability'
]

supplier_score = supplier_score.merge(
    price_std,
    on='VendorNumber'
)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

cols_to_scale = [
    'risk_score',
    'payment_delay',
    'processing_delay',
    'recent_bad_rate',
    'vendor_prev_delay',
    'price_variability'
]

supplier_score[cols_to_scale] = (
    scaler.fit_transform(
        supplier_score[cols_to_scale]
    )
)

In [ ]:
supplier_score['final_score'] = (
    0.50 * supplier_score['risk_score'] +
    0.15 * supplier_score['payment_delay'] +
    0.10 * supplier_score['processing_delay'] +
    0.10 * supplier_score['recent_bad_rate'] +
    0.10 * supplier_score['vendor_prev_delay'] +
    0.05 * supplier_score['price_variability']
)

In [ ]:
supplier_score = supplier_score.sort_values(
    by='final_score'
)

In [ ]:
supplier_score['category'] = pd.cut(
    supplier_score['final_score'],
    bins=[0, 0.3, 0.6, 1],
    labels=['Preferred', 'Moderate', 'Risky']
)

In [ ]:
supplier_score[['VendorNumber','VendorName','final_score','category']].head(10